# Assignment 2: Transformer Language Models

In this assignment you will implement a Transformer-based language model following the **OLMo 2 architecture**, train it, and compare it to a pre-trained model.

![Olmo2 overview](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/olmo2_overview.svg)

**Task markers:**
- 🎓 Suitable for oral exam discussion
- ⚙ Pure implementation task

---

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/hoda/anaconda3/envs/tch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


---

## Part 1: Building the Transformer Components

### Task 1.1 — MLP Layer ⚙

Implement the **SwiGLU** MLP using `hidden_size` and `intermediate_size` hyperparameters.

![SwiGLU](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/swiglu.svg)

SwiGLU uses element-wise multiplication (⊗) between two linear branches:

$$\text{MLP}(x) = \bigl(W_1 x \cdot \text{SiLU}(W_2 x)\bigr) W_3$$

All `nn.Linear` layers should use `bias=False`.

**Sanity check:** Create an untrained MLP layer. Create a 3-dimensional tensor where the last dimension equals `hidden_size`. Applying the MLP to this tensor should produce output with the same shape as the input.

In [2]:
# Task 1.1 — SwiGLU MLP

class A2MLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        # TODO: define linear layers (bias=False)
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj   = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        # TODO: implement SwiGLU forward pass
        # MLP(x) = (W1(x) * SiLU(W2(x))) @ W3
        return self.down_proj(self.gate_proj(x) * F.silu(self.up_proj(x)))


# Sanity check
mlp = A2MLP(hidden_size=64, intermediate_size=128)
x = torch.randn(2, 10, 64)
assert mlp(x).shape == x.shape, f"Expected {x.shape}, got {mlp(x).shape}"
print("Task 1.1 sanity check passed:", mlp(x).shape)


Task 1.1 sanity check passed: torch.Size([2, 10, 64])


### Task 1.2 — Normalization ⚙

Implement **Root Mean Square (RMS) layer normalization**, or use PyTorch's built-in `nn.RMSNorm`.

Configuration:
- `eps` → `rms_norm_eps`
- `normalized_shape` → hidden layer size
- `elementwise_affine=True`

**Sanity check:** Apply the same testing approach as Task 1.1.

In [3]:
# Task 1.2 — RMS Normalization

class A2RMSNorm(nn.Module):
    def __init__(self, hidden_size, rms_norm_eps=1e-5):
        super().__init__()
        # TODO
        self.norm = nn.RMSNorm(
            normalized_shape=hidden_size,
            eps=rms_norm_eps,
            elementwise_affine=True,
        )

    def forward(self, x):
        # TODO
        return self.norm(x)


# Sanity check
norm = A2RMSNorm(hidden_size=64)
x = torch.randn(2, 10, 64)
assert norm(x).shape == x.shape, f"Expected {x.shape}, got {norm(x).shape}"
print("Task 1.2 sanity check passed:", norm(x).shape)


Task 1.2 sanity check passed: torch.Size([2, 10, 64])


### Task 1.3 — Multi-Head Attention 🎓

Implement standard **multi-head attention** with **RoPE** (Rotary Position Embedding).

![MHA](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/mha.svg)

Key hyperparameters: `hidden_size`, `num_attention_heads`  
Derived: `d_h = hidden_size // num_attention_heads`

**Steps:**
1. Compute query, key, value projections
2. Reshape: `q = q.view(b, m, n_h, d_h).transpose(1, 2)`
3. Apply RoPE via the provided `apply_rotary_pos_emb` utility
4. Compute attention using `F.scaled_dot_product_attention` with `is_causal=True`
5. Project output

$$\alpha(q, k) = \frac{q \cdot k^T}{\sqrt{d_h}}$$
$$A(q, k) = \text{softmax}(\alpha(q, k) + \text{mask})$$
$$\text{Attention}(q, k, v) = A(q, k) \cdot v$$

All projection layers: `bias=False`.

**Sanity check:** Verify output shape and no crashes at multiple intermediate stages.

In [4]:
# Provided utility — RoPE helper (do not modify)
def apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [5]:
# Task 1.3 — Multi-Head Attention with RoPE

class A2RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        # TODO: precompute inverse frequencies and register as buffer
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_position_embeddings = max_position_embeddings

    def forward(self, x, seq_len=None):
        # TODO: return cos, sin
        seq_len = seq_len or x.shape[-2]
        t = torch.arange(seq_len, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)             # (seq_len, dim/2)
        emb = torch.cat([freqs, freqs], dim=-1)           # (seq_len, dim)
        cos = emb.cos().unsqueeze(0).unsqueeze(0)         # (1, 1, seq_len, dim)
        sin = emb.sin().unsqueeze(0).unsqueeze(0)
        return cos, sin


class A2MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_attention_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_attention_heads
        self.head_dim = hidden_size // num_attention_heads
        # TODO: define q, k, v, and output projections (bias=False)
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x, rotary_emb):
        b, m, _ = x.shape
        q = self.q_proj(x).view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, m, self.num_heads, self.head_dim).transpose(1, 2)
        cos, sin = rotary_emb(x, seq_len=m)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        attn_output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_output = attn_output.transpose(1, 2).reshape(b, m, self.hidden_size)
        return self.o_proj(attn_output)  # TODO


# Sanity check
attn = A2MultiHeadAttention(hidden_size=64, num_attention_heads=4)
rope = A2RotaryEmbedding(dim=16)
x = torch.randn(2, 10, 64)
assert attn(x, rope).shape == x.shape, f"Expected {x.shape}, got {attn(x, rope).shape}"
print("Task 1.3 sanity check passed:", attn(x, rope).shape)


Task 1.3 sanity check passed: torch.Size([2, 10, 64])


### Task 1.4 — Full Transformer Decoder Layer 🎓

Assemble a single Transformer decoder block. In `__init__`, create:
- Multi-head attention layer
- MLP layer
- Two RMSNorm normalizers

![fullblock](https://raw.githubusercontent.com/ricj/dsai-nlp.github.io/refs/heads/master/_pages/dat450/fullblock.svg)

In `forward`, connect them with **residual connections** at the right places:

```
h_new = do_something(h_old)
out   = h_new + h_old
```

**Sanity check:** Verify correct output shapes and no crashes.

In [6]:
# Task 1.4 — Transformer Decoder Layer

class A2TransformerLayer(nn.Module):
    def __init__(self, hidden_size, num_attention_heads, intermediate_size, rms_norm_eps=1e-5):
        super().__init__()
        # TODO: attention, mlp, norms
        self.attn_norm = A2RMSNorm(hidden_size, rms_norm_eps)
        self.attn      = A2MultiHeadAttention(hidden_size, num_attention_heads)
        self.mlp_norm  = A2RMSNorm(hidden_size, rms_norm_eps)
        self.mlp       = A2MLP(hidden_size, intermediate_size)

    def forward(self, x, rotary_emb):
        # TODO: apply norm → attention → residual, then norm → mlp → residual
        x = x + self.attn(self.attn_norm(x), rotary_emb)
        x = x + self.mlp(self.mlp_norm(x))
        return x


# Sanity check
layer = A2TransformerLayer(hidden_size=64, num_attention_heads=4, intermediate_size=128)
rope = A2RotaryEmbedding(dim=16)
x = torch.randn(2, 10, 64)
assert layer(x, rope).shape == x.shape, f"Expected {x.shape}, got {layer(x, rope).shape}"
print("Task 1.4 sanity check passed:", layer(x, rope).shape)


Task 1.4 sanity check passed: torch.Size([2, 10, 64])


### Task 1.5 — Complete Transformer Stack 🎓

Assemble the full model including:
- Token embedding layer
- Stack of Transformer decoder layers (use `nn.ModuleList`, not a plain Python list)
- Final RMSNorm
- Unembedding (LM head) layer — **no bias terms**

Create `A2RotaryEmbedding` in `__init__` and pass the rotations to each layer in `forward`.

**Sanity check:** Create a 2-dimensional *integer* tensor and apply your Transformer to it. The result should be a 3-dimensional tensor where the last dimension equals the vocabulary size.

In [7]:
# Task 1.5 — Full Transformer Stack

from types import SimpleNamespace

class A2TransformerConfig:
    def __init__(
        self,
        vocab_size=32000,
        hidden_size=256,
        num_hidden_layers=4,
        num_attention_heads=4,
        intermediate_size=512,
        rms_norm_eps=1e-5,
        max_position_embeddings=2048,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.rms_norm_eps = rms_norm_eps
        self.max_position_embeddings = max_position_embeddings


class A2TransformerModel(nn.Module):
    def __init__(self, config: A2TransformerConfig):
        super().__init__()
        # TODO: embedding, ModuleList of layers, norm, lm_head (bias=False)
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            A2TransformerLayer(
                config.hidden_size,
                config.num_attention_heads,
                config.intermediate_size,
                config.rms_norm_eps,
            )
            for _ in range(config.num_hidden_layers)
        ])
        self.norm    = A2RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.rotary_emb = A2RotaryEmbedding(
            dim=config.hidden_size // config.num_attention_heads,
            max_position_embeddings=config.max_position_embeddings,
        )

    def forward(self, input_ids, labels=None):
        # TODO
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x, self.rotary_emb)
        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].reshape(-1, logits.shape[-1])
            shift_labels = labels[:, 1:].reshape(-1)
            loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)

        return SimpleNamespace(loss=loss, logits=logits)


# Sanity check
config = A2TransformerConfig()
model = A2TransformerModel(config)
x = torch.randint(0, config.vocab_size, (2, 20))
out = model(x)
assert out.logits.shape == (2, 20, config.vocab_size), \
    f"Expected (2, 20, {config.vocab_size}), got {out.logits.shape}"
print("Task 1.5 sanity check passed:", out.logits.shape)


Task 1.5 sanity check passed: torch.Size([2, 20, 32000])


---

## Part 2: Training

### Task 2.1 — Training the Language Model 🎓

Select suitable hyperparameters (number of Transformer layers, hidden size, number of attention heads).  
For this assignment, use a **small Transformer** (e.g. a couple of layers).

Run the training function and compute the perplexity on the validation set (same method as Assignment 1).

> **Alternative:** Use the HuggingFace `Trainer`.

In [8]:
# Task 2.1 — Setup: device, dataset, tokenizer, DataLoaders

import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load wiki data (same files as Assignment 1)
TRAIN_FILE = "data/wiki.train.tokens"
VAL_FILE   = "data/wiki.valid.tokens"

dataset = load_dataset("text", data_files={"train": TRAIN_FILE, "val": VAL_FILE})
dataset = dataset.filter(lambda x: x["text"].strip() != "")

# Use a small training subset for speed; keep full validation set
dataset["train"] = Subset(dataset["train"], range(5000))

print(f"Train: {len(dataset['train'])} | Val: {len(dataset['val'])}")

# BPE tokenizer — SmolLM2-135M is locally cached
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")


Device: cuda


Train: 5000 | Val: 2461


Tokenizer vocab size: 49152


In [9]:
# Task 2.1 — Train the Transformer

BATCH_SIZE = 16
MAX_LEN    = 128

def collate_fn(batch):
    texts = [item["text"] for item in batch]
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    return {"input_ids": input_ids, "labels": labels}

# TODO: training loop (reuse or adapt A1Trainer)
# Small Transformer: 2 layers, hidden=256, 4 heads
config = A2TransformerConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=256,
    num_hidden_layers=2,
    num_attention_heads=4,
    intermediate_size=512,
)
model = A2TransformerModel(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
dl_train  = DataLoader(dataset["train"], batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
dl_val    = DataLoader(dataset["val"],   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

N_EPOCHS = 5
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in dl_train:
        optimizer.zero_grad()
        out = model(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += out.loss.item()
    train_loss /= len(dl_train)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in dl_val:
            val_loss += model(**batch).loss.item()
    val_loss /= len(dl_val)

    print(f"Epoch {epoch}/{N_EPOCHS}  train_loss={train_loss:.4f}  "
          f"val_loss={val_loss:.4f}  val_ppl={np.exp(val_loss):.1f}")


Model parameters: 26,477,824


Epoch 1/5  train_loss=6.4708  val_loss=5.3323  val_ppl=206.9


Epoch 2/5  train_loss=5.2753  val_loss=5.0862  val_ppl=161.8


Epoch 3/5  train_loss=4.8197  val_loss=4.9790  val_ppl=145.3


Epoch 4/5  train_loss=4.4797  val_loss=4.9210  val_ppl=137.1


Epoch 5/5  train_loss=4.1932  val_loss=4.9038  val_ppl=134.8


---

## Part 3: Text Generation

### Task 3.1 — Predicting the Next Word ⚙

Apply the model to an encoded prompt, extract the output at the **last position**, find the highest-scoring token index with `argmax`, and decode it using the tokenizer.

In [10]:
# Task 3.1 — Next-word prediction

def predict_next_word(model, tokenizer, prompt, topk=5):
    dev = next(model.parameters()).device
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(dev)

    model.eval()
    with torch.no_grad():
        out = model(input_ids=input_ids)

    # Logits at the last position predict the next token
    last_logits = out.logits[0, -1, :]
    top_values, top_indices = last_logits.topk(topk)
    probs = torch.softmax(top_values, dim=0)
    return [(tokenizer.decode([idx.item()]), round(prob.item(), 4))
            for idx, prob in zip(top_indices, probs)]


# Test
# print(predict_next_word(model, tokenizer, "The patient was diagnosed with"))


### Task 3.2 — Generating Texts 🎓

Implement a **random sampling** algorithm with the following parameters:
- `model`
- `prompt`
- `max_length`
- `temperature`
- `topk`

Terminate when the end-of-text symbol is produced or `max_length` steps are reached.  
Use `torch.distributions.Categorical` for sampling and `torch.topk` for top-K filtering.

Experiment with different values of `temperature` and `topk` and observe the effect on output quality.

> **Answer:** Lower temperature like 0.3 makes the output repetitive, the model keeps picking the same likely words. Higher temperature like 1.5 makes it very random and the text stops making sense. For topk, small values like 5 can make the model get stuck in a loop, while large values like 100 add variety but also noise. Temperature around 0.8 and topk around 40 seemed to give the best results for this model.

In [11]:
# Task 3.2 — Text generation with top-K sampling

def generate_text(model, tokenizer, prompt, max_length=100, temperature=1.0, topk=50):
    # Hint: use torch.distributions.Categorical for sampling
    # dist = torch.distributions.Categorical(logits=filtered_logits)
    # next_token = dist.sample()
    dev = next(model.parameters()).device
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(dev)  # (1, seq_len)

    model.eval()
    with torch.no_grad():
        for _ in range(max_length):
            out = model(input_ids=input_ids)
            next_logits = out.logits[0, -1, :] / temperature   # (vocab,)

            # Top-K filtering: zero out all but the top-k logits
            top_values, _ = torch.topk(next_logits, topk)
            threshold = top_values[-1]
            filtered_logits = next_logits.masked_fill(next_logits < threshold, float("-inf"))

            dist = torch.distributions.Categorical(logits=filtered_logits)
            next_token = dist.sample().unsqueeze(0).unsqueeze(0)  # (1, 1)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


# Test with sample prompts
# print(generate_text(model, tokenizer, "The patient was diagnosed with"))
# print(generate_text(model, tokenizer, "The study found that", temperature=0.7, topk=20))


### Task 3.3 — Comparing to a Pre-trained Transformer 🎓

Load the pre-trained **OLMo-2 1B** model and compare its text generation quality to your own model.

> **Note:** This model is *not* instruction-tuned — use it as a language model only.

> **Optional:** Copy weights from the pre-trained model into your implementation to verify architectural equivalence.

> **Answer:** The gap is obvious. My model outputs random text with `<unk>` tokens and jumps between topics. For the patient diagnosis prompt it generates something about ships and hockey teams which makes no sense. OLMo-2 1B stays on topic and produces text that actually fits the prompt. My model has 26M parameters trained on a small subset, OLMo-2 has 1B parameters trained on a much larger corpus, so the difference is expected.

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'allenai/OLMo-2-0425-1B'

# Download or point to a local cache directory
local_dir = model_name   # change to local path if already downloaded

# Task 3.3 — Load pre-trained model and tokenizer
tokenizer_pretrained = AutoTokenizer.from_pretrained(local_dir)
model_pretrained = AutoModelForCausalLM.from_pretrained(local_dir, torch_dtype=torch.float16)
model_pretrained = model_pretrained.to(device).eval()

# Wrap model_pretrained so generate_text can call model(input_ids=...)
class HFModelWrapper(nn.Module):
    def __init__(self, hf_model):
        super().__init__()
        self.hf_model = hf_model
    def forward(self, input_ids, labels=None):
        out = self.hf_model(input_ids=input_ids, labels=labels)
        return SimpleNamespace(loss=out.loss, logits=out.logits)

pretrained_wrapped = HFModelWrapper(model_pretrained)

# Compare generation quality
prompts = [
    "The patient was diagnosed with",
    "The study found that",
    "In the beginning of the 20th century,",
]
print("=== Small trained Transformer ===")
for p in prompts:
    print(f"  [{p}]")
    print(f"  → {generate_text(model, tokenizer, p, temperature=0.8, topk=40)}\n")

print("=== Pre-trained OLMo-2 1B ===")
for p in prompts:
    print(f"  [{p}]")
    print(f"  → {generate_text(pretrained_wrapped, tokenizer_pretrained, p, temperature=0.8, topk=40)}\n")


`torch_dtype` is deprecated! Use `dtype` instead!


2026-06-09 17:08:15.461351: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

=== Small trained Transformer ===
  [The patient was diagnosed with]


  → The patient was diagnosed with 3 @.@ 55 million ( 1 @.@ 6 m ) long , and moved into a half @-@ season . She was the two @-@ 0 in ( 1999 ) 4 in ( 1 @.@ 9 @.@ 5 mi ) . Other ships , the Blue Jackets and 1549 . In the 2015 , <unk> the city was completed in 19

  [The study found that]
  → The study found that the album is home to the album 's Day , it is a number of all of the second album , to <unk> and <unk> <unk> . 2010 , the " I 's , and the " I was a few population " and " . In 2008 , she was written by <unk> and <unk> <unk> <unk> , who would have been on a <unk> of The Feast of <unk> ,

  [In the beginning of the 20th century,]


  → In the beginning of the 20th century,@ 000 kg ( 2012 ) . A deep and 7 February 131 @,@ 000 sq mi ) of the University of 1 @,@ 000 to 136 2 <unk> <unk> @-@ <unk> <unk> <unk> . The <unk> 77 @,@ <unk> <unk> <unk> 25 m ( 12

=== Pre-trained OLMo-2 1B ===
  [The patient was diagnosed with]


  → The patient was diagnosed with a paroxysmal hystero-epilepsy with 1 h periods of generalized seizure activity. Following seizure activity, no neurological signs were observed. A single seizure with generalized tonic-clonic activity was observed. Neurology consultations were performed, but no seizures were observed during subsequent hospitalizations. No antiepileptic drug intake was observed during the hospitalizations in both settings.
During the hospitalization in the second setting, the patient was prescribed a single 40 mg/day dose of carbam

  [The study found that]


  → The study found that a child's brain is not fully developed until the age of 17.  Also, while the brain is developing, it is being exposed to a lot of different stimuli.  This exposure to new information is important because it is helping the brain to form connections between different parts of the brain.  The study authors also found that children with ADHD were more likely to have "weak connections between different regions of the brain".  This could be due to the fact that ADHD children have

  [In the beginning of the 20th century,]


  → In the beginning of the 20th century, the industrial and financial interests in the United States began to push for the use of the dollar in international trade and in the expansion of their trade with China. In 1913, U.S. Federal Reserve President Nelson Aldridge Marriner Eccles reported to Congress that the “dollar was no longer a hard metal, it was a paper currency.”

In 1924, President Calvin Coolidge declared the dollar an “official standard of payment and value.” In 1934, Congress passed legislation to

